# Interactive k-Medoids Walkthrough (Lecture 11)

Step through a small k-medoids example under the L1 distance and inspect candidate medoid costs.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))

DATA = np.array([[1, 8], [2, 7], [3, 8], [4, 7], [5, 6], [6, 5], [7, 3], [8, 4], [8, 2]], dtype=float)


def l1(a, b):
    return np.abs(a - b).sum(axis=-1)


def assign(X, M):
    d = np.abs(X[:, None, :] - M[None, :, :]).sum(axis=2)
    return d.argmin(axis=1)


def best_medoid(X, labels, j):
    idx = np.where(labels == j)[0]
    scores = []
    for i in idx:
        cost = np.abs(X[idx] - X[i]).sum(axis=1).sum()
        scores.append((i, cost))
    scores.sort(key=lambda z: z[1])
    return scores


def stage_state(m1=3, m2=8, stage=0):
    M = DATA[[m1, m2]].copy()
    labels = assign(DATA, M)
    cand = best_medoid(DATA, labels, 0)
    if stage >= 2:
        cand2 = best_medoid(DATA, labels, 1)
        M = DATA[[cand[0][0], cand2[0][0]]].copy()
        labels = assign(DATA, M)
        cand = cand2
    return M, labels, cand


def draw(m1=3, m2=8, stage=0):
    M, labels, cand = stage_state(m1, m2, stage)
    fig, axs = plt.subplots(1, 2, figsize=(12.8, 4.5))
    axs[0].scatter(DATA[:, 0], DATA[:, 1], c=labels, s=60, cmap="tab10")
    for i, p in enumerate(DATA):
        axs[0].text(p[0] + 0.08, p[1] + 0.08, f"x{i+1}", fontsize=9)
    axs[0].scatter(M[:, 0], M[:, 1], c="k", marker="D", s=120)
    axs[0].set_title("k-medoids step walkthrough")
    axs[0].grid(alpha=0.2)
    rows = cand[:5]
    axs[1].axis("off")
    table = axs[1].table(
        cellText=[[f"x{i+1}", f"{cost}"] for i, cost in rows],
        colLabels=["candidate", "L1 cost"],
        loc="center",
        cellLoc="center",
    )
    table.scale(1, 1.4)
    axs[1].set_title("Candidate medoid costs")
    plt.show()

widgets.interact(
    draw,
    m1=widgets.IntSlider(min=0, max=8, step=1, value=3, continuous_update=False),
    m2=widgets.IntSlider(min=0, max=8, step=1, value=8, continuous_update=False),
    stage=widgets.IntSlider(min=0, max=5, step=1, value=0, continuous_update=False),
)
